<a href="https://colab.research.google.com/github/JasonL888/AI_Experiments/blob/main/RAG_Agent/rag_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Custom RAG

## Pre-requisites
- update `.env` with `HF_TOKEN` from HuggingFace
- upload your PDFs to `./pdfs`

In [ ]:
# likely for local env - may not be needed for Colab
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 transformers sentence-transformers

In [2]:
!pip install chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 11.6 MB/s eta

In [3]:
import os
import glob

from typing import List, Dict, Tuple
from dotenv import load_dotenv

import pypdf
import chromadb
from chromadb.utils import embedding_functions

from transformers import pipeline # For LLM pipeline

In [4]:
import sys
from getpass import getpass

# ensure pdf folder exists
PDF_DIR = "pdfs"
os.makedirs(PDF_DIR, exist_ok=True)

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Colab: prompt for file upload (files.upload returns dict of filename->bytes)
    from google.colab import files
    print("Detected Google Colab. Use the file picker to upload your PDFs.")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        dest = os.path.join(PDF_DIR, filename)
        with open(dest, "wb") as f:
            f.write(content)
        print(f"Saved uploaded file to: {dest}")

# HF_TOKEN handling: prefer environment or .env, otherwise prompt
load_dotenv()
hf = os.environ.get("HF_TOKEN")

if not hf:
    # Colab: ask for HF token (hidden) and persist to .env
    if IN_COLAB:
        token = getpass("Enter HuggingFace HF_TOKEN (hidden): ")
        if token:
            os.environ["HF_TOKEN"] = token
            # persist token for future cells (writes .env in workspace)
            with open(".env", "a") as envf:
                envf.write(f"\nHF_TOKEN={token}\n")
            print("HF_TOKEN set for session and appended to .env")
    else:
        # Local (VS Code): check .env then prompt if missing
        if os.path.exists(".env"):
            load_dotenv(".env")
            hf = os.environ.get("HF_TOKEN")
        if not hf:
            print("HF_TOKEN not found. Create a .env file with HF_TOKEN=your_token or paste it now.")
            token = getpass("Paste HF_TOKEN (hidden): ")
            if token:
                os.environ["HF_TOKEN"] = token
                with open(".env", "a") as envf:
                    envf.write(f"\nHF_TOKEN={token}\n")
                print("HF_TOKEN written to .env")

print(f"PDF directory: {os.path.abspath(PDF_DIR)}")

Detected Google Colab. Use the file picker to upload your PDFs.


Saving ABC_Bank_FAQ.pdf to ABC_Bank_FAQ.pdf
Saved uploaded file to: pdfs/ABC_Bank_FAQ.pdf
Enter HuggingFace HF_TOKEN (hidden): ··········
HF_TOKEN set for session and appended to .env
PDF directory: /content/pdfs


In [5]:
# --- Configuration ---
load_dotenv()

# Directory where your PDF files are stored.
PDF_DIR = "pdfs"

# Model for creating embeddings (converts text into vectors).
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Local LLM model for the RAG generation phase.
FLAN_T5_MODEL = "google/flan-t5-small"

# Name of the Chroma collection (the vector store database).
COLLECTION_NAME = "pdf_rag_collection"
CHROMA_DB_PATH = "./chroma_db"

# Text splitting parameters
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

In [6]:
def split_text_to_chunks(text: str, source_path: str, page_num: int) -> List[Tuple[str, Dict]]:
    """
    A simple text splitter function to break down large documents.
    This mimics recursive splitting logic using simple overlap.
    Returns a list of (text_chunk, metadata) tuples.
    """
    chunks = []
    i = 0
    while i < len(text):
        end = i + CHUNK_SIZE
        chunk = text[i:end]

        # Add chunk and metadata
        metadata = {
            "source": source_path,
            "page": page_num,
            "chunk_index": len(chunks)
        }
        chunks.append((chunk, metadata))

        # Determine the next starting point
        i += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks

In [7]:
def initialize_llm_pipeline():
    """
    Initializes and returns the HuggingFace LLM pipeline using the specified model.
    """
    try:
        print(f"\n--- Initializing HuggingFace Model: {FLAN_T5_MODEL} ---")
        llm_pipeline = pipeline(
            "text2text-generation",
            model=FLAN_T5_MODEL
        )
        print("Model loaded successfully.")
        return llm_pipeline
    except ImportError:
        print("\nError: Required libraries ('transformers' or 'torch') not found.")
        print("Please install them: 'pip install transformers torch'")
        return None
    except Exception as e:
        print(f"Error loading model {FLAN_T5_MODEL}: {e}")
        return None

In [8]:
def load_and_index_pdfs(directory_path: str) -> chromadb.Collection:
    """
    Loads all PDF files, chunks them, and indexes them into a Chroma collection.

    Returns:
        A configured Chroma collection object.
    """
    print(f"--- Step 1 & 2: Loading and Chunking Documents from '{directory_path}' ---")

    if not os.path.isdir(directory_path):
        print(f"Error: Directory '{directory_path}' not found. Please create it.")
        return None

    pdf_files = glob.glob(os.path.join(directory_path, "*.pdf"))
    if not pdf_files:
        print(f"Warning: No PDF files found in '{directory_path}'.")
        return None

    all_chunks: List[Tuple[str, Dict]] = []
    total_pages = 0

    for file_path in pdf_files:
        try:
            reader = pypdf.PdfReader(file_path)
            total_pages += len(reader.pages)

            for i, page in enumerate(reader.pages):
                page_text = page.extract_text() or ""

                # Use custom chunking logic
                page_chunks = split_text_to_chunks(page_text, file_path, i)
                all_chunks.extend(page_chunks)

            print(f"Processed: {os.path.basename(file_path)} ({len(reader.pages)} pages)")

        except Exception as e:
            print(f"Could not process {file_path}: {e}")

    if not all_chunks:
        print("No readable content found. Exiting indexing.")
        return None

    print(f"Total pages processed: {total_pages}. Created {len(all_chunks)} text chunks.")

    # --- Step 3: Initializing Embeddings and Vector Store (Chromadb Native) ---
    print(f"\n--- Step 3: Creating Embeddings and Indexing (Model: {EMBEDDING_MODEL}) ---")

    try:
        # Initialize Chroma Client
        client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

        # Initialize Sentence Transformer for embedding generation
        embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=EMBEDDING_MODEL,
            device='cpu' # Use CPU for local environments
        )

        # Create or retrieve the collection
        collection = client.get_or_create_collection(
            name=COLLECTION_NAME,
            embedding_function=embedding_function
        )

        # Prepare data for native Chroma addition
        texts = [chunk[0] for chunk in all_chunks]
        metadatas = [chunk[1] for chunk in all_chunks]
        ids = [f"doc_{i}" for i in range(len(texts))]

        # Clear existing data if necessary (optional)
        existing_ids = collection.get().get('ids', [])
        if existing_ids:
            collection.delete(ids=existing_ids)
        else:
            print("No existing documents to delete from collection.")

        # Add data to the collection
        collection.add(
            documents=texts,
            metadatas=metadatas,
            ids=ids
        )

        print(f"Indexing complete. Total documents in collection: {collection.count()}")
        return collection

    except Exception as e:
        print(f"An error occurred during embedding or vector store creation: {e}")
        return None

In [9]:
def invoke_llm_with_context(llm_pipeline, context: str, query: str) -> str:
    """
    Uses a HuggingFace pipeline to generate a response based on the context.

    (The content of this function remains largely the same, only the input
     source for the context changes from LangChain Document to raw string.)
    """
    if llm_pipeline is None:
        return "LLM pipeline is not initialized. Cannot generate response."

    print("\n--- LLM Invocation (HuggingFace Pipeline) ---")

    # Construct the RAG prompt template for the LLM
    system_prompt = (
        "You are an intelligent assistant. Use the following context to answer the question. "
        "Provide a concise, detailed answer. "
        "If the answer cannot be found in the context, state politely that the information is not available in the provided documents."
    )
    prompt = f"{system_prompt}\n\nContext: {context}\n\nQuestion: {query}\n\nAnswer:"

    # Generate the response
    try:
        result = llm_pipeline(prompt, max_new_tokens=256, clean_up_tokenization_spaces=True)
        generated_text = result[0]['generated_text']

        return generated_text.strip()

    except Exception as e:
        return f"Error during LLM generation: {e}"

In [10]:
def rag_query(vectorstore_collection: chromadb.Collection, prompt: str, llm_pipeline):
    """
    Performs the RAG retrieval and LLM generation using the native Chroma collection.

    Args:
        vectorstore_collection: The configured Chroma collection object.
        prompt: The user's question.
        llm_pipeline: The initialized HuggingFace pipeline object.
    """
    # --- Step 4: Retrieval (Chromadb Native) ---
    print(f"\n--- Step 4: Retrieval for Query: '{prompt}' ---")

    # Perform the semantic search
    results = vectorstore_collection.query(
        query_texts=[prompt],
        n_results=4, # Retrieve top 4 documents/chunks
        include=['documents', 'metadatas'] # Ask for the content and source info
    )

    # Extract documents (context) and metadata
    retrieved_documents = results.get('documents', [[]])[0]
    retrieved_metadatas = results.get('metadatas', [[]])[0]

    # Combine the content of the retrieved documents into a single context string
    context = "\n\n---\n\n".join(retrieved_documents)

    print(f"Found {len(retrieved_documents)} relevant chunks.")

    # Display the sources
    print("--- Sources ---")
    for i, metadata in enumerate(retrieved_metadatas):
        source = metadata.get('source', 'Unknown Source')
        # pypdf page numbers are 0-indexed, display as 1-indexed for user readability
        page = metadata.get('page', 0) + 1
        print(f"Chunk {i+1}: Source: {os.path.basename(source)}, Page: {page}")

    # --- Step 5: Generation ---
    final_response = invoke_llm_with_context(llm_pipeline, context, prompt)

    print("\n" + "="*50)
    print("FINAL RAG RESULT:")
    print(final_response)
    print("="*50)

In [11]:
print("="*50)
print("Python RAG System for PDF Indexing and Querying")
print("="*50)

# 1. Setup Phase: Load, Chunk, and Index
vector_db = load_and_index_pdfs(PDF_DIR)

if vector_db:
    # Initialize the LLM pipeline
    llm = initialize_llm_pipeline()

    # 2. Query Phase: Use the RAG system
    user_query = input("\nEnter your query based on the PDF content (e.g., 'What are the three main steps of the process?'):\n> ")

    if user_query:
        rag_query(vector_db, user_query, llm)
    else:
        print("Query skipped.")
else:
    print("\nCould not initialize the vector store. Please check the directory and PDF files.")

Python RAG System for PDF Indexing and Querying
--- Step 1 & 2: Loading and Chunking Documents from 'pdfs' ---
Processed: ABC_Bank_FAQ.pdf (21 pages)
Total pages processed: 21. Created 71 text chunks.

--- Step 3: Creating Embeddings and Indexing (Model: sentence-transformers/all-MiniLM-L6-v2) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Model loaded successfully.

Enter your query based on the PDF content (e.g., 'What are the three main steps of the process?'):
> what are the benefits of savings account


Token indices sequence length is longer than the specified maximum sequence length for this model (667 > 512). Running this sequence through the model will result in indexing errors



--- Step 4: Retrieval for Query: 'what are the benefits of savings account' ---
Found 4 relevant chunks.
--- Sources ---
Chunk 1: Source: ABC_Bank_FAQ.pdf, Page: 17
Chunk 2: Source: ABC_Bank_FAQ.pdf, Page: 7
Chunk 3: Source: ABC_Bank_FAQ.pdf, Page: 15
Chunk 4: Source: ABC_Bank_FAQ.pdf, Page: 15

--- LLM Invocation (HuggingFace Pipeline) ---

FINAL RAG RESULT:
allowing your money to grow faster. These accounts are ideal for individuals looking to maximize savings without the risk of investing. Interest compounds monthly, enhancing your returns over time. High-yield savings accounts typically have higher balance requirements, but they provide easy access to funds. Contact us to learn about current rates and account minimums. • Does ABC Bank offer fraud protection on debit and credit cards? --- to build comprehensive plans, including budgeting, investing, and retirement planning. Financial plans are tailored to meet specific objectives, from short-term savings to long-term wealth growth.